[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C10_Eval_Measurement_Course/07_online_ab_eval/07_online_ab_eval.ipynb)

# 07 · 在线 A/B 评测（从零实现）

目标：把在线实验的核心统计手艺**纯 numpy 从零实现**，并用 **scipy 对拍**。① 两比例 z 检验 + Welch t 检验；② 功效模拟与样本量；③ 多重比较（Bonferroni/BH）；④ 偷看如何抬高假阳性 + α-spending 校正；⑤ CUPED 方差缩减。

路线：z 检验 → Welch t 检验 → 功效模拟 → 样本量 → 多重比较 → CUPED → ✏️ 练习 → 📖 答案 → 🧪 真实数据（UCI Adult）A/B 胶囊。

> 纪律：自己实现的每个检验都与 `scipy.stats` **对拍到 1e-6**；所有 Monte-Carlo 性质（功效、假阳性率）用足够多次试验 + `assert` 验证；随机用 `default_rng(seed)`。

## 0 · 数据 helper（联网取真实数据，失败回退）

下面这个 cell 定义全课统一的下载工具，最后的真实数据胶囊会用到。

In [ ]:
import os, json, urllib.request, re
import numpy as np
import pandas as pd
CACHE = os.path.expanduser('~/.eval_measurement_data'); os.makedirs(CACHE, exist_ok=True)

def _get(url, fn=None, timeout=30):
    if fn:
        path = os.path.join(CACHE, fn)
        if not os.path.exists(path):
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            open(path, 'wb').write(urllib.request.urlopen(req, timeout=timeout).read())
        return path
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    return urllib.request.urlopen(req, timeout=timeout).read()

def hf_rows(dataset, config, split, n=300, fn=None):
    fn = fn or f"{dataset.replace('/', '_')}_{config}_{split}_{n}.json"
    path = os.path.join(CACHE, fn)
    if os.path.exists(path):
        return json.load(open(path))
    out = []; off = 0
    while len(out) < n:
        L = min(100, n - len(out))
        u = (f'https://datasets-server.huggingface.co/rows?dataset={dataset.replace("/", "%2F")}'
             f'&config={config}&split={split}&offset={off}&length={L}')
        r = json.loads(_get(u)); rows = [x['row'] for x in r['rows']]
        if not rows: break
        out += rows; off += L
    json.dump(out, open(path, 'w')); return out

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)
print('数据 helper 就绪；缓存目录 =', CACHE)

## 1 · 两比例 z 检验（从零 + 对拍）

比例型指标（转化率、点击率）的 A/B 检验。$H_0$：两组真实比例相等。用合并比例估方差：

$$z=\frac{\hat p_B-\hat p_A}{\sqrt{\hat p(1-\hat p)(1/n_A+1/n_B)}},\quad p=2(1-\Phi(|z|))$$

从零实现，再与 `scipy.stats.norm` 直接计算对拍。

In [ ]:
import numpy as np
from scipy import stats
rng = np.random.default_rng(0)

def two_proportion_ztest(x_a, n_a, x_b, n_b):
    '''x=转化数, n=样本数。返回 (z, p_two_sided)。'''
    p_a, p_b = x_a / n_a, x_b / n_b
    p_pool = (x_a + x_b) / (n_a + n_b)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
    z = (p_b - p_a) / se
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return z, p

# 对照 11.8% (n=8000) vs 处理 13.2% (n=8000)
x_a, n_a = 944, 8000
x_b, n_b = 1056, 8000
z, p = two_proportion_ztest(x_a, n_a, x_b, n_b)
print(f'转化率 A={x_a/n_a:.4f}  B={x_b/n_b:.4f}')
print(f'z = {z:.4f},  p = {p:.4f}')
# 对拍: 用 erf 独立算一遍 p
from math import erf, sqrt
p_check = 2 * (1 - 0.5 * (1 + erf(abs(z) / sqrt(2))))
assert abs(p - p_check) < 1e-10, 'p 值两种算法必须一致'
print(f'对拍 erf 算 p = {p_check:.4f}  (一致 ✅)')
assert p < 0.05, '这个差异在 n=8000 下应当显著'
print('✅ 两比例 z 检验从零实现正确，p 值与独立计算对拍一致')

## 2 · Welch t 检验（从零 + 对拍 scipy）

连续型指标（人均时长/收入）的 A/B 检验。Welch 不假设两组等方差（线上默认）：

$$t=\frac{\bar Y_B-\bar Y_A}{\sqrt{s_A^2/n_A+s_B^2/n_B}}$$

自由度用 Welch–Satterthwaite。从零实现，**与 `scipy.stats.ttest_ind(equal_var=False)` 对拍到 1e-6**。

In [ ]:
def welch_ttest(a, b):
    '''两样本 Welch t 检验。返回 (t, p_two_sided, df)。'''
    a, b = np.asarray(a, float), np.asarray(b, float)
    na, nb = len(a), len(b)
    ma, mb = a.mean(), b.mean()
    va, vb = a.var(ddof=1), b.var(ddof=1)   # 样本方差 (ddof=1)
    se = np.sqrt(va / na + vb / nb)
    t = (mb - ma) / se
    # Welch-Satterthwaite 自由度
    df = (va/na + vb/nb)**2 / ((va/na)**2/(na-1) + (vb/nb)**2/(nb-1))
    p = 2 * stats.t.sf(abs(t), df)          # 双尾
    return t, p, df

a = rng.normal(10.0, 3.0, 500)            # 对照: 人均时长
b = rng.normal(10.6, 3.2, 520)            # 处理: 略高
t, p, df = welch_ttest(a, b)
t_sp, p_sp = stats.ttest_ind(b, a, equal_var=False)   # 注意顺序 b,a 对应 mb-ma
print(f'自己实现 : t={t:.6f}  p={p:.6f}  df={df:.2f}')
print(f'scipy    : t={t_sp:.6f}  p={p_sp:.6f}')
assert abs(t - t_sp) < 1e-6, 't 必须与 scipy 一致'
assert abs(p - p_sp) < 1e-6, 'p 必须与 scipy 一致'
print('✅ Welch t 检验从零实现，与 scipy.stats.ttest_ind 对拍到 1e-6')

## 3 · 功效模拟：真有效应时能检出的概率

**功效 = 1−β = 真有效应时拒绝 H0 的概率**。最可靠的算法是蒙特卡洛：固定真效应，反复跑 A/B，统计显著比例。

两个验证：① 真效应大 + 样本足 → 功效高；② **零效应 → 拒绝率 ≈ α**（这验证了第一类错误受控）。

In [ ]:
def simulate_power(mean_a, mean_b, sd, n, alpha=0.05, trials=3000, seed=0):
    '''蒙特卡洛估功效: 每次抽两组正态样本跑 Welch t, 返回 p<alpha 的比例。'''
    r = np.random.default_rng(seed)
    rej = 0
    for _ in range(trials):
        a = r.normal(mean_a, sd, n)
        b = r.normal(mean_b, sd, n)
        _, p, _ = welch_ttest(a, b)
        if p < alpha:
            rej += 1
    return rej / trials

# 情形1: 有效应 (10.0 vs 10.8, sd=3, n=400) -> 功效应较高
power = simulate_power(10.0, 10.8, 3.0, 400, trials=3000, seed=1)
print(f'有效应(Δ=0.8, n=400): 模拟功效 = {power:.3f}')
# 解析功效对拍 (正态近似)
delta = 0.8; se = 3.0 * np.sqrt(2/400)
z_alpha = stats.norm.ppf(1 - 0.05/2)
analytic = 1 - stats.norm.cdf(z_alpha - delta/se) + stats.norm.cdf(-z_alpha - delta/se)
print(f'解析功效(正态近似)     = {analytic:.3f}')
assert abs(power - analytic) < 0.04, '模拟功效应接近解析功效'

# 情形2: 零效应 -> 拒绝率应 ≈ alpha=0.05 (第一类错误受控)
fpr = simulate_power(10.0, 10.0, 3.0, 400, trials=4000, seed=2)
print(f'\n零效应: 拒绝率 = {fpr:.3f}  (应 ≈ α=0.05)')
assert abs(fpr - 0.05) < 0.02, '零效应下拒绝率应约等于 alpha'
print('✅ 功效模拟: 有效应时功效与解析一致, 零效应时第一类错误≈α')

## 4 · 样本量计算：要看清这么小的效应，要多少流量？

给定 MDE、α、目标功效，反解每组样本量：

$$n\approx\frac{2\sigma^2(z_{1-\alpha/2}+z_{1-\beta})^2}{\Delta^2}$$

算出 n 后，用模拟在这个 n 上验证达到了目标功效。

In [ ]:
def sample_size_per_arm(mde, sd, alpha=0.05, power=0.8):
    '''两样本均值检验所需每组样本量。'''
    z_a = stats.norm.ppf(1 - alpha/2)
    z_b = stats.norm.ppf(power)
    n = 2 * sd**2 * (z_a + z_b)**2 / mde**2
    return int(np.ceil(n))

mde, sd = 0.5, 3.0
n = sample_size_per_arm(mde, sd, alpha=0.05, power=0.8)
print(f'要以 80% 功效检出 Δ=0.5 (sd=3): 每组需 n = {n}')
# 验证: 在这个 n 上模拟, 功效应 ≈ 0.8
achieved = simulate_power(10.0, 10.0 + mde, sd, n, trials=3000, seed=3)
print(f'在 n={n} 上模拟得到的功效 = {achieved:.3f}  (目标 0.8)')
assert abs(achieved - 0.8) < 0.05, '达到的功效应接近目标 0.8'
# 效应减半 -> 样本量应约 4 倍
n_half = sample_size_per_arm(mde/2, sd, power=0.8)
print(f'效应减半 (Δ=0.25): 每组需 n = {n_half}  (约 {n_half/n:.1f}× )')
assert 3.5 < n_half / n < 4.5, '效应减半样本量应约 4 倍'
print('✅ 样本量公式正确: 模拟功效达标, 且效应减半→样本量4倍')

## 5 · 多重比较：假阳性泛滥与校正

做 m 个**都无效应**的检验, 至少一个假阳性的概率 = 1−(1−α)^m。

演示: 不校正时假阳性泛滥; **Bonferroni**(阈值 α/m, 控 FWER) 和 **Benjamini-Hochberg**(控 FDR) 把它压回来。

In [ ]:
def benjamini_hochberg(pvals, alpha=0.05):
    '''BH 步进法控制 FDR。返回被拒绝的布尔数组。'''
    pvals = np.asarray(pvals)
    m = len(pvals)
    order = np.argsort(pvals)
    sorted_p = pvals[order]
    # 找最大的 k 使 p_(k) <= (k/m)*alpha  (k 从 1 计)
    thresh = (np.arange(1, m + 1) / m) * alpha
    below = sorted_p <= thresh
    k = np.max(np.where(below)[0]) + 1 if below.any() else 0
    reject = np.zeros(m, dtype=bool)
    if k > 0:
        reject[order[:k]] = True
    return reject

# 1) 全是无效应: 看 FWER
def family_fp_rate(m, corrected, trials=2000, alpha=0.05, seed=0):
    r = np.random.default_rng(seed)
    any_fp = 0
    for _ in range(trials):
        # m 个无效应检验: 每个的 p 在 H0 下 ~ Uniform(0,1)
        ps = r.uniform(0, 1, m)
        if corrected == 'none':
            rej = ps < alpha
        elif corrected == 'bonferroni':
            rej = ps < alpha / m
        elif corrected == 'bh':
            rej = benjamini_hochberg(ps, alpha)
        any_fp += rej.any()
    return any_fp / trials

m = 20
fwer_none = family_fp_rate(m, 'none', seed=1)
fwer_bonf = family_fp_rate(m, 'bonferroni', seed=1)
print(f'm={m} 个无效应检验, 至少一个假阳性的概率:')
print(f'  不校正    : {fwer_none:.3f}  (理论 1-(1-α)^m = {1-(1-0.05)**m:.3f})')
print(f'  Bonferroni: {fwer_bonf:.3f}  (应被压回 ≈ α=0.05)')
assert fwer_none > 0.4, '不校正时 FWER 应远超 α'
assert fwer_bonf < 0.08, 'Bonferroni 应把 FWER 压回 α 附近'
print('✅ 多重比较: 不校正假阳性泛滥(64%), Bonferroni 控回 5%')

**BH 在有真效应时比 Bonferroni 多发现真效应**：混入若干真效应，比较两者的检出数与 FDR。

In [ ]:
# 2) 混入真效应: 20 个检验, 前 6 个有强真效应(p 很小), 后 14 个无效应
def make_pvalues(n_true=6, n_null=14, seed=0):
    r = np.random.default_rng(seed)
    p_true = r.uniform(0, 0.001, n_true)      # 强效应 -> 极小 p
    p_null = r.uniform(0, 1, n_null)          # 无效应 -> 均匀 p
    ps = np.concatenate([p_true, p_null])
    is_true = np.array([True]*n_true + [False]*n_null)
    return ps, is_true

ps, is_true = make_pvalues(seed=5)
rej_bonf = ps < 0.05 / len(ps)
rej_bh = benjamini_hochberg(ps, 0.05)
print(f'真效应数 = {is_true.sum()}')
print(f'Bonferroni 检出 {rej_bonf.sum()} 个 (真阳性 {(rej_bonf & is_true).sum()})')
print(f'BH         检出 {rej_bh.sum()} 个 (真阳性 {(rej_bh & is_true).sum()})')
assert rej_bh.sum() >= rej_bonf.sum(), 'BH 应检出 >= Bonferroni'
# BH 的 FDR 应受控: 假阳性 / 总拒绝 <= 0.05 (这组数据下)
fdr_bh = (rej_bh & ~is_true).sum() / max(rej_bh.sum(), 1)
print(f'BH 的 FDR = {fdr_bh:.3f}  (控制在 α=0.05 附近)')
assert fdr_bh <= 0.05 + 1e-9, 'BH 应控制 FDR'
print('✅ BH 控 FDR 的同时, 比保守的 Bonferroni 多发现真效应')

## 6 · CUPED：用实验前协变量免费缩方差

$Y^{cuped}=Y-\theta(X-\bar X)$, $\theta=Cov(Y,X)/Var(X)$, X 是**实验前**协变量。

两个性质要验证: ① **无偏**(均值不变); ② **方差按 1−ρ² 缩小**(ρ=corr(X,Y))。

In [ ]:
def cuped_adjust(Y, X):
    '''用实验前协变量 X 调整 Y。返回调整后指标。'''
    theta = np.cov(Y, X, ddof=1)[0, 1] / np.var(X, ddof=1)
    return Y - theta * (X - X.mean())

# 造 X(实验前), Y(实验中), 相关 rho
rho = 0.7
n = 20000
X = rng.normal(0, 1, n)
noise = rng.normal(0, 1, n)
Y = rho * X + np.sqrt(1 - rho**2) * noise + 10.0   # corr(X,Y)≈rho, mean≈10

Y_cuped = cuped_adjust(Y, X)
print(f'原始    : 均值={Y.mean():.4f}  方差={Y.var(ddof=1):.4f}')
print(f'CUPED后 : 均值={Y_cuped.mean():.4f}  方差={Y_cuped.var(ddof=1):.4f}')
# ① 无偏: 均值几乎不变
assert abs(Y_cuped.mean() - Y.mean()) < 0.02, 'CUPED 应无偏(均值不变)'
# ② 方差缩减 ≈ 1-rho^2
var_ratio = Y_cuped.var(ddof=1) / Y.var(ddof=1)
print(f'方差比 = {var_ratio:.3f}  (理论 1-ρ² = {1-rho**2:.3f})')
assert abs(var_ratio - (1 - rho**2)) < 0.03, '方差应按 1-rho^2 缩小'
print(f'✅ CUPED: 无偏 + 方差缩小 {(1-var_ratio)*100:.0f}% (相当于样本量约 {1/var_ratio:.1f}×)')

**CUPED 用在 A/B 上 = 提升功效**：对调整后的指标做 t 检验，在同样数据上比原始指标更容易检出真效应。

In [ ]:
# 同一个 A/B, 比较用原始 Y vs 用 CUPED-Y 的功效
def ab_power_with_cuped(use_cuped, rho=0.7, effect=0.1, n=1500, trials=2000, seed=0):
    r = np.random.default_rng(seed)
    rej = 0
    for _ in range(trials):
        Xa = r.normal(0, 1, n); Xb = r.normal(0, 1, n)
        Ya = rho*Xa + np.sqrt(1-rho**2)*r.normal(0,1,n) + 10.0
        Yb = rho*Xb + np.sqrt(1-rho**2)*r.normal(0,1,n) + 10.0 + effect
        if use_cuped:
            # 用合并数据估 theta (实务做法)
            Xall = np.concatenate([Xa, Xb]); Yall = np.concatenate([Ya, Yb])
            theta = np.cov(Yall, Xall, ddof=1)[0,1] / np.var(Xall, ddof=1)
            mu = Xall.mean()
            Ya = Ya - theta*(Xa - mu); Yb = Yb - theta*(Xb - mu)
        _, pval, _ = welch_ttest(Ya, Yb)
        rej += pval < 0.05
    return rej / trials

pow_raw = ab_power_with_cuped(False, seed=7)
pow_cuped = ab_power_with_cuped(True, seed=7)
print(f'同一 A/B (Δ=0.1, n=1500):')
print(f'  用原始指标   功效 = {pow_raw:.3f}')
print(f'  用 CUPED指标 功效 = {pow_cuped:.3f}')
assert pow_cuped > pow_raw + 0.05, 'CUPED 应显著提升功效'
print('✅ CUPED 在不加样本的情况下显著提升了检出真效应的功效')

---
## ✏️ 练习 1：单尾 vs 双尾 + 两比例检验

实现 `ztest_onesided(x_a, n_a, x_b, n_b)`：检验 $H_1: p_B > p_A$（单尾）。

返回 `(z, p_one_sided)`，其中单尾 `p = 1 - Φ(z)`（只在 B 更大方向算）。验证：同样数据下单尾 p 约为双尾 p 的一半（当 z>0）。

In [ ]:
from scipy import stats
def ztest_onesided(x_a, n_a, x_b, n_b):
    # TODO: 算合并比例 p_pool, 标准误 se, z=(p_b-p_a)/se
    #       单尾 p = 1 - Φ(z)  (检验 B>A); 返回 (z, p)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
x_a, n_a, x_b, n_b = 944, 8000, 1008, 8000
z1, p1 = ztest_onesided(x_a, n_a, x_b, n_b)
# 双尾对照
p_pool = (x_a+x_b)/(n_a+n_b)
se = np.sqrt(p_pool*(1-p_pool)*(1/n_a+1/n_b))
z_ref = (x_b/n_b - x_a/n_a)/se
p2 = 2*(1-stats.norm.cdf(abs(z_ref)))
assert abs(z1 - z_ref) < 1e-9, 'z 应与参考一致'
assert abs(p1 - p2/2) < 1e-9, '当 z>0 时单尾 p 应为双尾的一半'
print(f'z={z1:.4f}  单尾 p={p1:.4f}  双尾 p={p2:.4f}')
print('✅ 练习 1 通过：单尾检验 = 双尾的一半(同向)')

## ✏️ 练习 2：比例指标的样本量

实现 `sample_size_proportion(p_base, mde_abs, alpha=0.05, power=0.8)`：检出从 `p_base` 到 `p_base+mde_abs` 的**绝对**提升所需每组样本量。

用 $n\approx\frac{(z_{1-\alpha/2}+z_{1-\beta})^2\,[p_1(1-p_1)+p_2(1-p_2)]}{\text{mde}^2}$。验证：基线越接近 0.5（方差越大）需要越多样本。

In [ ]:
def sample_size_proportion(p_base, mde_abs, alpha=0.05, power=0.8):
    # TODO: p1=p_base, p2=p_base+mde_abs; z_a=norm.ppf(1-alpha/2), z_b=norm.ppf(power)
    #       n = (z_a+z_b)^2 * (p1*(1-p1)+p2*(1-p2)) / mde_abs^2; 返回 ceil(n) 的 int
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
n1 = sample_size_proportion(0.10, 0.02)   # 基线 10%
n2 = sample_size_proportion(0.50, 0.02)   # 基线 50% (方差最大)
print(f'基线 10%, 检出 +2%: 每组 n = {n1}')
print(f'基线 50%, 检出 +2%: 每组 n = {n2}')
assert isinstance(n1, int) and n1 > 0
assert n2 > n1, '基线越接近 0.5 (方差越大) 需要越多样本'
# 模拟验证 n1 达到 ~80% 功效
def sim_prop_power(p1, p2, n, trials=2500, seed=0):
    r = np.random.default_rng(seed); rej=0
    for _ in range(trials):
        xa = r.binomial(n, p1); xb = r.binomial(n, p2)
        pp=(xa+xb)/(2*n); se=np.sqrt(pp*(1-pp)*(2/n))
        z=(xb/n-xa/n)/se if se>0 else 0
        rej += 2*(1-stats.norm.cdf(abs(z)))<0.05
    return rej/trials
ap = sim_prop_power(0.10, 0.12, n1)
print(f'在 n={n1} 上模拟功效 = {ap:.3f} (目标≈0.8)')
assert abs(ap - 0.8) < 0.06, '应达到目标功效'
print('✅ 练习 2 通过：比例样本量正确, 基线 0.5 最贵')

## ✏️ 练习 3：偷看抬高假阳性 + α-spending 校正

模拟在 H0 为真（无效应）下，把数据分成 `n_looks` 批逐步累积，每批后检验一次，**任意一次显著就叫停**。

实现 `peeking_fpr(n_looks, alpha=0.05, correct=False, trials=2000, seed=0)`：返回叫停（误报）比例。`correct=False` 用阈值 α；`correct=True` 用 Bonferroni 校正阈值 `α/n_looks`。验证：不校正时 FPR ≫ α，校正后压回。

In [ ]:
def peeking_fpr(n_looks, alpha=0.05, correct=False, trials=2000, seed=0):
    r = np.random.default_rng(seed)
    thresh = alpha / n_looks if correct else alpha
    stopped = 0
    batch = 300                              # 每批每组样本数
    for _ in range(trials):
        a_all = np.array([]); b_all = np.array([])
        fired = False
        for _l in range(n_looks):
            # 累积一批 H0 数据(两组同分布)
            a_all = np.concatenate([a_all, r.normal(0, 1, batch)])
            b_all = np.concatenate([b_all, r.normal(0, 1, batch)])
            # TODO: 对当前累积的 a_all,b_all 跑 welch_ttest, 若 p<thresh 则 fired=True; break
            raise NotImplementedError
        stopped += fired
    return stopped / trials

In [ ]:
# —— 练习 3 自测 ——
fpr_peek = peeking_fpr(5, correct=False, seed=1)
fpr_corr = peeking_fpr(5, correct=True, seed=1)
print(f'偷看 5 次, 不校正: 假阳性率 = {fpr_peek:.3f}  (应 > α=0.05)')
print(f'偷看 5 次, α/5 校正: 假阳性率 = {fpr_corr:.3f}  (应被压低)')
assert fpr_peek > 0.08, '偷看应抬高假阳性率'
assert fpr_corr < fpr_peek, '校正应降低假阳性率'
print('✅ 练习 3 通过：偷看抬高假阳性, α-spending 把它压回来')

## ✏️ 练习 4：CUPED 的 θ 与方差缩减

实现 `cuped_theta_and_reduction(Y, X)`：返回 `(theta, var_reduction_ratio)`，其中 `theta = Cov(Y,X)/Var(X)`，`var_reduction_ratio = 1 - Var(Y_cuped)/Var(Y)`（缩减了百分之多少）。

验证：缩减比例 ≈ ρ²（ρ=corr(X,Y)）。

In [ ]:
def cuped_theta_and_reduction(Y, X):
    # TODO: theta=Cov(Y,X)/Var(X); Y_cuped=Y-theta*(X-mean(X))
    #       reduction = 1 - var(Y_cuped)/var(Y); 返回 (theta, reduction)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
rho = 0.6; n = 30000
rg = np.random.default_rng(11)
X = rg.normal(0, 1, n)
Y = rho*X + np.sqrt(1-rho**2)*rg.normal(0, 1, n) + 5.0
theta, reduction = cuped_theta_and_reduction(Y, X)
print(f'theta = {theta:.3f}  (≈斜率 ρ·sd_Y/sd_X ≈ {rho:.2f})')
print(f'方差缩减 = {reduction:.3f}  (理论 ρ² = {rho**2:.3f})')
assert abs(reduction - rho**2) < 0.03, '缩减比例应 ≈ ρ²'
assert abs(theta - rho) < 0.05, 'theta 应 ≈ ρ (此处 sd_X≈sd_Y≈1)'
print('✅ 练习 4 通过：CUPED 缩减方差 ≈ ρ²')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def ztest_onesided(x_a, n_a, x_b, n_b):
    p_a, p_b = x_a/n_a, x_b/n_b
    p_pool = (x_a+x_b)/(n_a+n_b)
    se = np.sqrt(p_pool*(1-p_pool)*(1/n_a+1/n_b))
    z = (p_b - p_a)/se
    p = 1 - stats.norm.cdf(z)
    return z, p

In [ ]:
# 练习 2 参考答案
def sample_size_proportion(p_base, mde_abs, alpha=0.05, power=0.8):
    p1 = p_base; p2 = p_base + mde_abs
    z_a = stats.norm.ppf(1 - alpha/2); z_b = stats.norm.ppf(power)
    n = (z_a + z_b)**2 * (p1*(1-p1) + p2*(1-p2)) / mde_abs**2
    return int(np.ceil(n))

In [ ]:
# 练习 3 参考答案
def peeking_fpr(n_looks, alpha=0.05, correct=False, trials=2000, seed=0):
    r = np.random.default_rng(seed)
    thresh = alpha / n_looks if correct else alpha
    stopped = 0; batch = 300
    for _ in range(trials):
        a_all = np.array([]); b_all = np.array([]); fired = False
        for _l in range(n_looks):
            a_all = np.concatenate([a_all, r.normal(0, 1, batch)])
            b_all = np.concatenate([b_all, r.normal(0, 1, batch)])
            _, p, _ = welch_ttest(a_all, b_all)
            if p < thresh:
                fired = True; break
        stopped += fired
    return stopped / trials

In [ ]:
# 练习 4 参考答案
def cuped_theta_and_reduction(Y, X):
    theta = np.cov(Y, X, ddof=1)[0, 1] / np.var(X, ddof=1)
    Y_cuped = Y - theta * (X - X.mean())
    reduction = 1 - Y_cuped.var(ddof=1) / Y.var(ddof=1)
    return theta, reduction

---
## 🧪 真实数据胶囊：真实数据上的两样本检验 + CUPED

用 **UCI Adult** 数据集（真实人口普查数据）做一个 A/B 风格分析。

> ⚠️ **这是观察性数据，不是随机实验**——我们把它当作「演示检验/CUPED 机制」的真实数据，而非因果结论。把 `sex` 当「分组」、`income>50K` 当二元指标做两比例检验；再用 `education_num` 当协变量演示 CUPED。

**联网取真实 Adult 数据；失败则回退到内置的真实风格数值。**

In [ ]:
def load_adult(n=4000):
    '''取真实 UCI Adult; 失败回退内置真实风格数据。返回 (df, source)。'''
    cols = ['age','workclass','fnlwgt','education','education_num','marital',
            'occupation','relationship','race','sex','capital_gain','capital_loss',
            'hours_per_week','country','income']
    try:
        path = _get('https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data', 'adult.data')
        df = pd.read_csv(path, header=None, names=cols, skipinitialspace=True)
        df = df.dropna().head(n)
        if len(df) > 100:
            return df, 'online'
    except Exception as e:
        print('  (联网失败，回退内置:', type(e).__name__, ')')
    # 回退: 内置真实风格数据(男性 income>50K 比例约 0.30, 女性约 0.11; 与真实 Adult 一致)
    rg = np.random.default_rng(0)
    n_each = n // 2
    sex = np.array(['Male']*n_each + ['Female']*n_each)
    edu = np.clip(rg.normal(10, 2.5, n).round(), 1, 16).astype(int)
    base = np.where(sex=='Male', 0.30, 0.11) + 0.02*(edu-10)/6   # 教育正相关
    inc = (rg.random(n) < np.clip(base, 0.01, 0.95))
    income = np.where(inc, '>50K', '<=50K')
    df = pd.DataFrame({'sex': sex, 'education_num': edu, 'income': income})
    return df, 'builtin'

df, src = load_adult(4000)
print(f'数据来源 = {src}; {len(df)} 行')
print(df[['sex','education_num','income']].head())
assert len(df) > 100
print('✅ 拿到真实(或回退) Adult 数据')

In [ ]:
# 两比例检验: 男 vs 女 的 income>50K 比例
high = (df['income'] == '>50K').values
is_male = (df['sex'] == 'Male').values
x_m, n_m = high[is_male].sum(), is_male.sum()
x_f, n_f = high[~is_male].sum(), (~is_male).sum()
z, p = two_proportion_ztest(x_f, n_f, x_m, n_m)
print(f'男性 income>50K 比例 = {x_m/n_m:.3f} (n={n_m})')
print(f'女性 income>50K 比例 = {x_f/n_f:.3f} (n={n_f})')
print(f'两比例 z 检验: z={z:.3f}, p={p:.2e}')
assert n_m > 10 and n_f > 10
print('（这是观察性差异, 不可解读为因果——仅演示检验机制在真实数据上的运行）')
print('✅ 真实数据两比例检验跑通')

**🧪 胶囊练习**：用 `education_num`（真实协变量）做 CUPED，缩减「`income>50K` 数值化指标」的方差。

实现 `cuped_on_real(df)`：把 `income>50K` 转成 0/1 数组 Y、`education_num` 当 X，返回 `(原始方差, CUPED后方差)`，CUPED 后方差应更小（因为教育与收入相关）。

In [ ]:
def cuped_on_real(df):
    # TODO: Y = (df['income']=='>50K').astype(float).values; X = df['education_num'].astype(float).values
    #       theta=Cov(Y,X)/Var(X); Y_cuped=Y-theta*(X-X.mean())
    #       返回 (Y.var(ddof=1), Y_cuped.var(ddof=1))
    raise NotImplementedError

In [ ]:
# 自测
v_raw, v_cuped = cuped_on_real(df)
print(f'原始指标方差   = {v_raw:.5f}')
print(f'CUPED后方差    = {v_cuped:.5f}  (缩减 {(1-v_cuped/v_raw)*100:.1f}%)')
assert v_cuped <= v_raw, 'CUPED 后方差不应增大'
print('✅ 胶囊练习通过：真实协变量(教育)缩减了收入指标的方差')

In [ ]:
# 📖 胶囊参考答案
def cuped_on_real(df):
    Y = (df['income'] == '>50K').astype(float).values
    X = df['education_num'].astype(float).values
    theta = np.cov(Y, X, ddof=1)[0, 1] / np.var(X, ddof=1)
    Y_cuped = Y - theta * (X - X.mean())
    return float(Y.var(ddof=1)), float(Y_cuped.var(ddof=1))

### 小结
- A/B 实验靠**随机化**让组间差异 = 因果效应；核心问题：**这点差别是真本事还是抽样噪声**。
- 两样本检验 = **信号/噪声**：比例用两比例 z、连续用 Welch t（都与 scipy 对拍）。
- **功效分析**先于实验：$n\propto\sigma^2/\Delta^2$，效应减半→样本量 4 倍；功效不足的「不显著」无信息。
- **多重比较**：检验做多了假阳性泛滥；Bonferroni 控 FWER（保守）、BH 控 FDR（多发现真效应）。
- **偷看**抬高假阳性；预定样本量或用 α-spending/always-valid。
- **CUPED** 用实验前协变量无偏地把方差按 1−ρ² 缩小 = 免费提功效；方差缩减意识对离线评测同样适用。

🎓 **全课终点**：你已经把评测从「跑个脚本打个分」升级为一门**测量科学**——从标签噪声、标注者一致性、生成指标、人评、IRT、校准到在线 A/B，每个数字你都知道它怎么来的、带多大误差、何时会骗人。回到 [课程主页](../index.html) 复习，或挑一个真实评测任务把这套手艺用起来。